In [1]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, find_subjects, preprocess_subject

cfg = load_config('../configs/eye_eeg_simul.yaml')
subjects = find_subjects(cfg)
print(f"Subjects to preprocess: {len(subjects)}")
print(subjects)

[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj2', 'subj20', 'subj28', 'subj42', 'subj7']
Subjects to preprocess: 30
['subj3', 'subj4', 'subj5', 'subj6', 'subj8', 'subj10', 'subj14', 'subj17', 'subj18', 'subj19', 'subj21', 'subj22', 'subj23', 'subj24', 'subj25', 'subj26', 'subj27', 'subj29', 'subj30', 'subj31', 'subj32', 'subj33', 'subj34', 'subj35', 'subj36', 'subj37', 'subj38', 'subj39', 'subj40', 'subj41']


In [2]:
# Test with the first subject
test_subject = subjects[0]
print(f"\nTesting preprocessing on: {test_subject}\n")

success = preprocess_subject(cfg, test_subject, overwrite=False, verbose=True)
print(f"\nResult: {'OK' if success else 'skipped or failed'}")


Testing preprocessing on: subj3

[subj3] preprocessing
   filter: 0.5-40.0 Hz (iir, order 4)
   EOG channels: none found in this recording (searched for: ['AUX_1', 'HEOG'])
   montage applied: standard_1020 (32 EEG channels)
   resample: 500.0 -> 256 Hz (factor 0.5120)
   saved: subj3_preprocessed_raw.fif
   saved: subj3_preprocessed_events_eve.fif

Result: OK


In [3]:
import mne
from eeg_toolkit import get_subject_path

raw_pp = mne.io.read_raw_fif(
    get_subject_path(cfg, test_subject, 'preprocessed_raw'),
    preload=False, verbose='WARNING'
)
events_pp = mne.read_events(
    get_subject_path(cfg, test_subject, 'preprocessed_events')
)

print(f"Channels: {len(raw_pp.ch_names)}")
print(f"Sample rate: {raw_pp.info['sfreq']} Hz")
print(f"Duration: {raw_pp.times[-1]:.1f} s")
print(f"Channel types: {set(raw_pp.get_channel_types())}")
print(f"Has montage: {raw_pp.info['dig'] is not None}")
print(f"\nEvents: {len(events_pp)}")
print(f"First 3 events:\n{events_pp[:3]}")

Channels: 32
Sample rate: 256.0 Hz
Duration: 2248.0 s
Channel types: {'eeg'}
Has montage: True

Events: 1680
First 3 events:
[[11696     0    50]
 [11846     0    20]
 [11975     0    51]]


In [4]:
from eeg_toolkit import preprocess_all

# Run preprocessing on all included subjects.
# subj3 will be skipped since it's already done (overwrite=False).
# Expect ~30-60 seconds per subject; total ~15-30 minutes.
summary = preprocess_all(cfg, overwrite=False, verbose=True)

[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj2', 'subj20', 'subj28', 'subj42', 'subj7']
Preprocessing 30 subject(s)

--- [1/30] subj3 ---
[subj3] already preprocessed — skipping (use overwrite=True to redo)

--- [2/30] subj4 ---
[subj4] preprocessing
   filter: 0.5-40.0 Hz (iir, order 4)
   EOG channels: none found in this recording (searched for: ['AUX_1', 'HEOG'])
   montage applied: standard_1020 (32 EEG channels)
   resample: 500.0 -> 256 Hz (factor 0.5120)
   saved: subj4_preprocessed_raw.fif
   saved: subj4_preprocessed_events_eve.fif

--- [3/30] subj5 ---
[subj5] preprocessing
   filter: 0.5-40.0 Hz (iir, order 4)
   EOG channels: none found in this recording (searched for: ['AUX_1', 'HEOG'])
   montage applied: standard_1020 (32 EEG channels)
   resample: 500.0 -> 256 Hz (factor 0.5120)
   saved: subj5_preprocessed_raw.fif
   saved: subj5_preprocessed_events_eve.fif

--- [4/30] subj6 ---
[subj6] preprocessing
   filter: 0.5-40.0 Hz (iir, order 4)
 

In [5]:
import mne
from eeg_toolkit import get_subject_path, load_config

cfg = load_config('../configs/eye_eeg_simul.yaml')
subj = 'subj3'  # mismo sujeto que preprocessaste

# Eventos antes del resample (500 Hz, post-XDF)
events_before = mne.read_events(get_subject_path(cfg, subj, 'events'))
sfreq_before = 500.0

# Eventos después del resample (256 Hz, post-preprocessing)
events_after = mne.read_events(get_subject_path(cfg, subj, 'preprocessed_events'))
sfreq_after = 256.0

# Comparar el primer evento en segundos
print(f"=== First event ===")
print(f"Before:  sample {events_before[0, 0]} @ {sfreq_before} Hz = {events_before[0, 0]/sfreq_before:.4f} s")
print(f"After:   sample {events_after[0, 0]} @ {sfreq_after} Hz = {events_after[0, 0]/sfreq_after:.4f} s")
print(f"Difference: {abs(events_before[0, 0]/sfreq_before - events_after[0, 0]/sfreq_after)*1000:.2f} ms")

print(f"\n=== Last event ===")
print(f"Before:  sample {events_before[-1, 0]} @ {sfreq_before} Hz = {events_before[-1, 0]/sfreq_before:.4f} s")
print(f"After:   sample {events_after[-1, 0]} @ {sfreq_after} Hz = {events_after[-1, 0]/sfreq_after:.4f} s")
print(f"Difference: {abs(events_before[-1, 0]/sfreq_before - events_after[-1, 0]/sfreq_after)*1000:.2f} ms")

# Verificación de la estructura del trial: distancia 50 -> 20 debe ser 500 ms
def get_first_trial_pattern(events, sfreq):
    code_50_idx = events[events[:, 2] == 50][0, 0]  # first fixation
    code_20_idx = events[events[:, 2] == 20][0, 0]  # first memory array
    return (code_20_idx - code_50_idx) / sfreq * 1000  # ms

ms_before = get_first_trial_pattern(events_before, sfreq_before)
ms_after  = get_first_trial_pattern(events_after,  sfreq_after)
print(f"\n=== Trial structure: fixation (50) → memory array (20) ===")
print(f"Before: {ms_before:.2f} ms (expected ~500 ms from PsychoPy script)")
print(f"After:  {ms_after:.2f} ms")

=== First event ===
Before:  sample 22844 @ 500.0 Hz = 45.6880 s
After:   sample 11696 @ 256.0 Hz = 45.6875 s
Difference: 0.50 ms

=== Last event ===
Before:  sample 1103107 @ 500.0 Hz = 2206.2140 s
After:   sample 564791 @ 256.0 Hz = 2206.2148 s
Difference: 0.84 ms

=== Trial structure: fixation (50) → memory array (20) ===
Before: 586.00 ms (expected ~500 ms from PsychoPy script)
After:  585.94 ms
